# GD GARFIN — Grenada Authority for the Regulation of Financial Institutions

Jira: DECD-4819. Source: https://garfin.gd/index.php/regulated-sectors

All four registers are static HTML (Joomla SP Page Builder tabs) — the entity lists are present in the page source, just hidden behind tabs. No Selenium/OCR needed: `requests` + BeautifulSoup is enough.

| ListCode | ListName | Tab(s) collected |
|---|---|---|
| 1 | Credit Unions | REGISTERED CREDIT UNIONS |
| 2 | Insurance Companies | REGISTERED COMPANIES + BROKERS |
| 3 | Money Services Businesses | MONEY SERVICES BUSINESSES |
| 4 | Pensions | REGISTERED PENSION PLANS (Active only) |

In [1]:
# ------------------------------------------------ Import Lib ----------------------------------------
import os
import re
import datetime
import requests
import pandas as pd
from bs4 import BeautifulSoup

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# ------------------------------------------------ Begin_fileName ----------------------------------------
regulatorName = 'GD GARFIN'  ## current controller name
print(f"Running {regulatorName} Web Scraping Tool v.1.0")

now = datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(':', '.')[:-7])

try:
    scriptfolder = os.path.dirname(os.path.abspath(__file__))  ## production environment (.py)
except NameError:
    scriptfolder = os.path.join(r"C:\Users\wuj1\OneDrive - Moody's\Desktop\Regulator", regulatorName)
os.chdir(scriptfolder)

processdate = now.strftime('%Y-%m-%d')

Running GD GARFIN Web Scraping Tool v.1.0


In [2]:
# ------------------------------------------------ Begin_Variable ----------------------------------------
sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [],
           'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [],
           'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [],
           'RegCtry': [], 'RegCode': [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
           'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [],
           'Phone - Mother company': []}

regdict = {
    regulatorName + ' 1': 'https://garfin.gd/index.php/regulated-sectors/credit-unions',
    regulatorName + ' 2': 'https://garfin.gd/index.php/regulated-sectors/insurance',
    regulatorName + ' 3': 'https://garfin.gd/index.php/regulated-sectors/money-services-businesses',
    regulatorName + ' 4': 'https://garfin.gd/index.php/regulated-sectors/pensions',
}

Typology = {
    regulatorName + ' 1': 'Credit Unions',
    regulatorName + ' 2': 'Insurance Companies',
    regulatorName + ' 3': 'Money Services Businesses',
    regulatorName + ' 4': 'Pensions',
}

# Tabs to collect per list (matched by the tab-nav link text)
tabsToCollect = {
    regulatorName + ' 1': ['REGISTERED CREDIT UNIONS'],
    regulatorName + ' 2': ['REGISTERED COMPANIES', 'BROKERS'],
    regulatorName + ' 3': ['MONEY SERVICES BUSINESSES'],
    regulatorName + ' 4': ['REGISTERED PENSION PLANS'],
}

HEADERS = {'User-Agent': 'Mozilla/5.0'}

# ------------------------------------------------ Begin_Function ----------------------------------------
def bourange_same_length_array(sqldict):
    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])
            for i in range(total_empty):
                empty.append('')
            sqldict[key] = sqldict[key] + empty
    return sqldict


def get_soup(url):
    r = requests.get(url, headers=HEADERS, timeout=30, verify=False)
    r.encoding = 'utf-8'
    return BeautifulSoup(r.text, 'html.parser')


def pane_by_tab(soup, tab_title):
    """Return the tab-pane <div> whose nav link text matches tab_title."""
    for a in soup.select('a[data-toggle="sppb-tab"]'):
        if a.get_text(strip=True).upper() == tab_title.upper():
            return soup.find(id=a.get('href', '').lstrip('#'))
    return None


def clean_names(li_list):
    out = []
    for li in li_list:
        t = li.get_text(' ', strip=True).replace('\xa0', ' ')
        t = re.sub(r'\s+', ' ', t).strip()
        if t:
            out.append(t)
    return out

In [ ]:
# ------------------------------------------------ Begin_Main ----------------------------------------
for reg in regdict:
    rows_before = len(sqldict['Name'])
    soup = get_soup(regdict[reg])

    names = []
    if reg.endswith(' 4'):
        # Pensions: collect only the entities listed under the 'Active' heading
        pane = pane_by_tab(soup, 'REGISTERED PENSION PLANS')
        active_hdr = pane.find('strong', string=lambda s: s and s.strip().lower() == 'active')
        active_ol = active_hdr.find_next('ol')
        names = clean_names(active_ol.find_all('li'))
    else:
        for tab_title in tabsToCollect[reg]:
            pane = pane_by_tab(soup, tab_title)
            names += clean_names(pane.find_all('li'))

    for name_ in names:
        sqldict['Name'].append(name_)
        sqldict['ListProcessDate'].append(processdate)
        sqldict['ListName'].append(Typology[reg])
        sqldict['RegCtry'].append(reg.split()[0])
        sqldict['RegCode'].append(reg.split()[1])
        sqldict['ListCode'].append(reg.split()[2])
        sqldict['Cntry'].append('GD')
        # sqldict['ListLanguage'].append('English')
        sqldict['RegulationType'].append('Regulated')
        sqldict = bourange_same_length_array(sqldict)

    rows_after = len(sqldict['Name'])
    print(f"[INFO] {reg} ({Typology[reg]}) collected {rows_after - rows_before} rows")

[INFO] GD GARFIN 1 (Credit Unions) collected 10 rows


[INFO] GD GARFIN 2 (Insurance Companies) collected 40 rows


[INFO] GD GARFIN 3 (Money Services Businesses) collected 10 rows


[INFO] GD GARFIN 4 (Pensions) collected 46 rows


In [4]:
# ------------------------------------------------ Begin_writer and save df to excel ----------------------------------------
os.chdir(scriptfolder)
df = pd.DataFrame(sqldict)
df = df[df['Name'] != '']
df.to_excel(filename, 'SQL Ready', index=False)
print('Saved {} rows -> {}'.format(len(df), filename))
df.groupby(['ListCode', 'ListName'])['Name'].count()

C:\Users\wuj1\AppData\Local\Temp\1\claude\ipykernel_18168\1664045245.py:5: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(filename, 'SQL Ready', index=False)


Saved 106 rows -> GD GARFIN SQL Ready 2026-06-05 10.32.51.xlsx


ListCode  ListName                 
1         Credit Unions                10
2         Insurance Companies          40
3         Money Services Businesses    10
4         Pensions                     46
Name: Name, dtype: int64